# AIE2026 Day2: official LoRA, 20 vs 10 epochs
Private coursework run. This upload contains official public notebook code, fictional FAQ examples, and a wrapper that saves actual metrics. No private research files, credentials, Slack content, or instructor metrics are used as completed training outputs.
Official course source: https://github.com/matsuolab/lecture-ai-engineering-2026/tree/fae899a409e2c99feee9d44a3789f936f5fc64ee
Two fresh Python processes; only num_train_epochs differs. GPU and internet are needed.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.18.0', 'peft==0.21.1', 'trl==1.14.2', 'accelerate==1.15.0', 'datasets==5.1.0'], check=True)


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, csv, shutil
ROOT = Path('/kaggle/working/aie2026-day2')
ROOT.mkdir(parents=True, exist_ok=True)
(ROOT/"official.ipynb").write_text('{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "id": "b5112f88",\n   "metadata": {},\n   "source": [\n    "# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する\\n",\n    "\\n",\n    "架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。\\n",\n    "\\n",\n    "**この演習で確かめること**\\n",\n    "\\n",\n    "1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。\\n",\n    "2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。\\n",\n    "3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。\\n",\n    "4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。\\n",\n    "\\n",\n    "**進め方**\\n",\n    "\\n",\n    "| セクション | 内容 | 所要時間の目安（T4） |\\n",\n    "| --- | --- | --- |\\n",\n    "| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |\\n",\n    "| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |\\n",\n    "| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |\\n",\n    "\\n",\n    "授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。\\n",\n    "\\n",\n    "**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。\\n",\n    "\\n",\n    "**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。\\n",\n    "\\n",\n    "版: 1.1（2026-10-18）"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "98a33c9e",\n   "metadata": {},\n   "source": [\n    "## 0. 環境の確認とライブラリの導入\\n",\n    "\\n",\n    "**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。\\n",\n    "\\n",\n    "**見るところ**\\n",\n    "\\n",\n    "- 1 つ目のセルに `Tesla T4` と表示されること。\\n",\n    "- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。\\n",\n    "\\n",\n    "**うまくいかないとき**\\n",\n    "\\n",\n    "- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。\\n",\n    "- pip の赤い `ERROR: pip\'s dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "dadb3b96",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "import subprocess\\n",\n    "try:\\n",\n    "    gpu_info = subprocess.run([\\"nvidia-smi\\", \\"--query-gpu=name,memory.total\\", \\"--format=csv\\"], capture_output=True, text=True).stdout\\n",\n    "except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い\\n",\n    "    gpu_info = \\"\\"\\n",\n    "print(gpu_info or \\"GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。\\")"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "4d946a72",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う\\n",\n    "PINS = {\\n",\n    "    \\"transformers\\": \\"5.18.0\\",   # Qwen3.5 は transformers 5 系が必要\\n",\n    "    \\"peft\\": \\"0.21.1\\",           # LoRA の実装\\n",\n    "    \\"trl\\": \\"1.14.2\\",            # 学習ループ（SFTTrainer）\\n",\n    "    \\"accelerate\\": \\"1.15.0\\",\\n",\n    "    \\"datasets\\": \\"5.1.0\\",\\n",\n    "}\\n",\n    "spec = \\" \\".join(f\'\\"{k}=={v}\\"\' for k, v in PINS.items())\\n",\n    "%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す\\n",\n    "%pip install -q {spec}\\n",\n    "\\n",\n    "# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい\\n",\n    "import transformers, peft, trl, accelerate, datasets, torch\\n",\n    "print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "bc7243bd",\n   "metadata": {},\n   "source": [\n    "## 1. 設定\\n",\n    "\\n",\n    "**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。\\n",\n    "\\n",\n    "- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。\\n",\n    "- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。\\n",\n    "- `gen: {\\"do_sample\\": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。\\n",\n    "\\n",\n    "`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。\\n",\n    "\\n",\n    "**見るところ**: 設定の JSON が表示されます。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "09c69560",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "from pathlib import Path\\n",\n    "import os, json, time\\n",\n    "\\n",\n    "CONFIG = {\\n",\n    "    \\"base_model\\": \\"Qwen/Qwen3.5-0.8B\\",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ\\n",\n    "    \\"revision\\": \\"2fc06364715b967f1860aea9cf38778875588b17\\",  # いつ実行しても同じ重みを使うために固定\\n",\n    "    \\"seed\\": 42,\\n",\n    "    \\"num_train_epochs\\": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）\\n",\n    "    \\"warmup_steps\\": 6,\\n",\n    "    \\"lr\\": 2e-4,\\n",\n    "    \\"lora_r\\": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）\\n",\n    "    \\"lora_alpha\\": 32,\\n",\n    "    \\"max_seq_len\\": 512,\\n",\n    "    \\"max_new_tokens\\": 96,\\n",\n    "    \\"gen\\": {\\"do_sample\\": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）\\n",\n    "    \\"work_dir\\": \\"/content/qlora_lab\\",                          # 結果ファイルの保存先（左のファイル一覧から見られる）\\n",\n    "    \\"drive_dir\\": \\"/content/drive/MyDrive/aie2026/qlora_lab\\",  # USE_DRIVE=True のときの保存先\\n",\n    "}\\n",\n    "WORK = Path(CONFIG[\\"work_dir\\"]); WORK.mkdir(parents=True, exist_ok=True)\\n",\n    "\\n",\n    "USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する\\n",\n    "if USE_DRIVE:\\n",\n    "    from google.colab import drive\\n",\n    "    drive.mount(\\"/content/drive\\")\\n",\n    "    Path(CONFIG[\\"drive_dir\\"]).mkdir(parents=True, exist_ok=True)\\n",\n    "print(json.dumps(CONFIG, ensure_ascii=False, indent=2))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "6c3e3709",\n   "metadata": {},\n   "source": [\n    "## 2. 架空 FAQ と教師データの生成\\n",\n    "\\n",\n    "**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。\\n",\n    "\\n",\n    "モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{\\"answer\\": <文字列>, \\"needs_review\\": <真偽値>}` の JSON 1 行です。\\n",\n    "\\n",\n    "- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。\\n",\n    "- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。\\n",\n    "\\n",\n    "**データの分け方**\\n",\n    "\\n",\n    "- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。\\n",\n    "- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。\\n",\n    "- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。\\n",\n    "- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。\\n",\n    "\\n",\n    "**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "9ea1a92c",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "FAQ = {\\n",\n    "    \\"deadline\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\",\\n",\n    "    \\"destination\\": \\"提出先は学習ポータルの「課題」欄です。\\",\\n",\n    "    \\"format\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\",\\n",\n    "    \\"office_hours\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\",\\n",\n    "    \\"late\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\",\\n",\n    "}\\n",\n    "FAQ_TEXT = \\"\\\\n\\".join(f\\"- {v}\\" for v in FAQ.values())\\n",\n    "\\n",\n    "# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)\\n",\n    "GROUNDED = {\\n",\n    "    \\"deadline\\": ([\\"課題の提出期限はいつですか\\", \\"いつまでに出せばいいですか\\", \\"締切を教えてください\\", \\"提出の期限は？\\"], [\\"9 月 30 日\\", \\"23:59\\"]),\\n",\n    "    \\"destination\\": ([\\"提出先はどこですか\\", \\"どこにアップロードしますか\\", \\"課題はどこから出しますか\\", \\"提出場所は？\\"], [\\"学習ポータル\\", \\"課題\\"]),\\n",\n    "    \\"format\\": ([\\"提出形式を教えてください\\", \\"ファイル名の決まりはありますか\\", \\"何のファイル形式で出しますか\\", \\"PDF 以外でも出せますか\\"], [\\"PDF\\"]),\\n",\n    "    \\"office_hours\\": ([\\"質問はいつできますか\\", \\"オフィスアワーはいつですか\\", \\"先生に質問する時間はありますか\\", \\"相談できる時間帯は？\\"], [\\"火曜\\", \\"17:00\\"]),\\n",\n    "    \\"late\\": ([\\"遅れて出したらどうなりますか\\", \\"期限を過ぎた場合は受け付けますか\\", \\"遅延提出のルールは？\\", \\"締切後の扱いを教えてください\\"], [\\"48 時間\\", \\"減点\\"]),\\n",\n    "}\\n",\n    "# 根拠なし: FAQ に無いので needs_review=true が正解\\n",\n    "UNGROUNDED = [\\n",\n    "    \\"再提出はできますか\\", \\"欠席した回の扱いを教えてください\\", \\"他の受講者の点数は見られますか\\",\\n",\n    "    \\"成績はいつ出ますか\\", \\"提出期限は延長できますか\\", \\"グループでの提出は可能ですか\\",\\n",\n    "    \\"課題の配点はどうなっていますか\\", \\"講義の録画は公開されますか\\",\\n",\n    "]\\n",\n    "\\n",\n    "# test 専用の追加質問（train / valid には入れない）\\n",\n    "TEST_EXTRA_GROUNDED = {\\n",\n    "    \\"deadline\\": [\\"提出期限の日付と時刻を教えてください\\", \\"課題の締切は何月何日の何時ですか\\"],\\n",\n    "    \\"destination\\": [\\"課題の提出はどこで行いますか\\", \\"提出物を出す先を教えてください\\"],\\n",\n    "    \\"format\\": [\\"提出するファイルの形式は何ですか\\", \\"提出ファイルは何形式で、いくつ出しますか\\"],\\n",\n    "    \\"office_hours\\": [\\"オフィスアワーの曜日と時間を教えてください\\", \\"質問を受け付けてもらえるのは何曜日の何時からですか\\"],\\n",\n    "    \\"late\\": [\\"提出が締切に間に合わなかったらどうなりますか\\", \\"期限後に提出した課題は受理されますか\\"],\\n",\n    "}\\n",\n    "TEST_EXTRA_UNGROUNDED = [\\n",\n    "    \\"課題の文字数に制限はありますか\\", \\"提出した課題へのフィードバックはもらえますか\\",\\n",\n    "    \\"参考文献の書き方に指定はありますか\\", \\"オフィスアワーは事前予約が必要ですか\\",\\n",\n    "]\\n",\n    "UNGROUNDED_ANSWER = \\"この FAQ には記載がありません。運営に確認が必要です。\\"\\n",\n    "\\n",\n    "def make_example(question: str, answer: str, needs_review: bool) -> dict:\\n",\n    "    target = json.dumps({\\"answer\\": answer, \\"needs_review\\": needs_review}, ensure_ascii=False)\\n",\n    "    return {\\"question\\": question, \\"faq\\": FAQ_TEXT, \\"target\\": target, \\"needs_review\\": needs_review}\\n",\n    "\\n",\n    "def split_templates(templates):\\n",\n    "    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）\\n",\n    "    return templates[:2], templates[2:3], templates[3:4]\\n",\n    "\\n",\n    "train, valid, test, gold = [], [], [], {}\\n",\n    "for key, (templates, must_include) in GROUNDED.items():\\n",\n    "    tr, va, te = split_templates(templates)\\n",\n    "    for q in tr: train.append(make_example(q, FAQ[key], False))\\n",\n    "    for q in va: valid.append(make_example(q, FAQ[key], False))\\n",\n    "    for q in te + TEST_EXTRA_GROUNDED[key]:\\n",\n    "        test.append(make_example(q, FAQ[key], False)); gold[q] = {\\"needs_review\\": False, \\"must_include\\": must_include}\\n",\n    "for i, q in enumerate(UNGROUNDED):\\n",\n    "    ex = make_example(q, UNGROUNDED_ANSWER, True)\\n",\n    "    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る\\n",\n    "    bucket.append(ex)\\n",\n    "    if bucket is test: gold[q] = {\\"needs_review\\": True, \\"must_include\\": [\\"確認\\"]}\\n",\n    "for q in TEST_EXTRA_UNGROUNDED:\\n",\n    "    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {\\"needs_review\\": True, \\"must_include\\": [\\"確認\\"]}\\n",\n    "\\n",\n    "# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める\\n",\n    "all_questions = [r[\\"question\\"] for r in train + valid + test]\\n",\n    "assert len(set(all_questions)) == len(all_questions), \\"質問が train / valid / test のどこかで重複しています。テンプレートを見直してください\\"\\n",\n    "print(f\\"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}\\")\\n",\n    "print(\\"学習例の 1 件 | 質問:\\", train[0][\\"question\\"], \\"| 望ましい出力:\\", train[0][\\"target\\"])\\n",\n    "_ = (WORK / \\"train.jsonl\\").write_text(\\"\\\\n\\".join(json.dumps(r, ensure_ascii=False) for r in train), encoding=\\"utf-8\\")\\n",\n    "_ = (WORK / \\"test.jsonl\\").write_text(\\"\\\\n\\".join(json.dumps(r, ensure_ascii=False) for r in test), encoding=\\"utf-8\\")\\n",\n    "_ = (WORK / \\"gold.json\\").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "98dd3f48",\n   "metadata": {},\n   "source": [\n    "## 3. モデルの読み込みと学習前評価\\n",\n    "\\n",\n    "**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。\\n",\n    "\\n",\n    "プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。\\n",\n    "\\n",\n    "**採点のしかた**（セクション 5 でも同じ関数を使います）\\n",\n    "\\n",\n    "- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。\\n",\n    "- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。\\n",\n    "\\n",\n    "内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。\\n",\n    "\\n",\n    "`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。\\n",\n    "\\n",\n    "**見るところ**\\n",\n    "\\n",\n    "- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。\\n",\n    "- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。\\n",\n    "\\n",\n    "**出ても問題ない警告**\\n",\n    "\\n",\n    "- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。\\n",\n    "  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。\\n",\n    "  - 自分のモデルやデータを Hub にアップロードするとき。\\n",\n    "  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。\\n",\n    "  \\n",\n    "  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。\\n",\n    "- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。\\n",\n    "\\n",\n    "**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "d7dc2471",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "import torch\\n",\n    "from transformers import AutoTokenizer, AutoModelForCausalLM\\n",\n    "\\n",\n    "DEVICE = \\"cuda\\" if torch.cuda.is_available() else (\\"mps\\" if torch.backends.mps.is_available() else \\"cpu\\")\\n",\n    "tok = AutoTokenizer.from_pretrained(CONFIG[\\"base_model\\"], revision=CONFIG[\\"revision\\"])\\n",\n    "if tok.pad_token is None: tok.pad_token = tok.eos_token\\n",\n    "# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）\\n",\n    "model = AutoModelForCausalLM.from_pretrained(CONFIG[\\"base_model\\"], revision=CONFIG[\\"revision\\"], dtype=torch.float32).to(DEVICE)\\n",\n    "model.config.use_cache = True\\n",\n    "BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)\\n",\n    "print(\\"loaded:\\", CONFIG[\\"base_model\\"], \\"| device:\\", DEVICE, \\"| 重みのメモリ(GB):\\", BASE_MEMORY_GB)\\n",\n    "\\n",\n    "# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点\\n",\n    "SYSTEM = \\"あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\\\\\\"answer\\\\\\": 文字列, \\\\\\"needs_review\\\\\\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。\\"\\n",\n    "\\n",\n    "def build_messages(example):\\n",\n    "    user = f\\"FAQ:\\\\n{example[\'faq\']}\\\\n\\\\n質問: {example[\'question\']}\\"\\n",\n    "    return [{\\"role\\": \\"system\\", \\"content\\": SYSTEM}, {\\"role\\": \\"user\\", \\"content\\": user}]\\n",\n    "\\n",\n    "def generate(model, example):\\n",\n    "    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）\\n",\n    "    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)\\n",\n    "    inputs = tok(prompt, return_tensors=\\"pt\\").to(model.device)\\n",\n    "    with torch.no_grad():\\n",\n    "        out = model.generate(**inputs, max_new_tokens=CONFIG[\\"max_new_tokens\\"], pad_token_id=tok.pad_token_id, **CONFIG[\\"gen\\"])\\n",\n    "    return tok.decode(out[0][inputs[\\"input_ids\\"].shape[1]:], skip_special_tokens=True).strip()"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "d9120ba6",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "import re\\n",\n    "import pandas as pd\\n",\n    "pd.set_option(\\"display.max_colwidth\\", None)   # 出力の原文を省略せずに表示する\\n",\n    "\\n",\n    "def check_format(text: str):\\n",\n    "    \'\'\'JSON として読め、必須キーと型が合うか。失敗理由も返す\'\'\'\\n",\n    "    try:\\n",\n    "        obj = json.loads(text)\\n",\n    "    except Exception as e:\\n",\n    "        return False, f\\"json_error: {type(e).__name__}\\", None\\n",\n    "    if not isinstance(obj, dict): return False, \\"not_object\\", None\\n",\n    "    if set(obj) - {\\"answer\\", \\"needs_review\\"}: return False, \\"extra_keys\\", obj\\n",\n    "    if \\"answer\\" not in obj or \\"needs_review\\" not in obj: return False, \\"missing_key\\", obj\\n",\n    "    if not isinstance(obj[\\"answer\\"], str) or not isinstance(obj[\\"needs_review\\"], bool): return False, \\"wrong_type\\", obj\\n",\n    "    return True, \\"\\", obj\\n",\n    "\\n",\n    "def check_content(obj, gold_entry):\\n",\n    "    \'\'\'内容の合否: needs_review が正解と一致し、必須語句を answer に含む\'\'\'\\n",\n    "    if obj is None: return False\\n",\n    "    if obj.get(\\"needs_review\\") != gold_entry[\\"needs_review\\"]: return False\\n",\n    "    return all(w in obj.get(\\"answer\\", \\"\\") for w in gold_entry[\\"must_include\\"])\\n",\n    "\\n",\n    "def mark(ok: bool) -> str:\\n",\n    "    return \\"○\\" if ok else \\"×\\"\\n",\n    "\\n",\n    "def evaluate(model, label: str):\\n",\n    "    \'\'\'test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する\'\'\'\\n",\n    "    rows = []\\n",\n    "    t0 = time.time()\\n",\n    "    for ex in test:\\n",\n    "        raw = generate(model, ex)\\n",\n    "        ok_fmt, reason, obj = check_format(raw)\\n",\n    "        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する\\n",\n    "        loose = obj\\n",\n    "        if not ok_fmt:\\n",\n    "            m = re.search(r\\"\\\\{.*\\\\}\\", raw, re.S)\\n",\n    "            try: loose = json.loads(m.group(0)) if m else None\\n",\n    "            except Exception: loose = None\\n",\n    "        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex[\\"question\\"]])\\n",\n    "        rows.append({\\"question\\": ex[\\"question\\"], \\"raw\\": raw, \\"format_ok\\": ok_fmt, \\"format_reason\\": reason,\\n",\n    "                     \\"content_ok\\": ok_content, \\"both_ok\\": ok_fmt and ok_content, \\"gold_needs_review\\": gold[ex[\\"question\\"]][\\"needs_review\\"]})\\n",\n    "    n = len(rows)\\n",\n    "    summary = {\\"label\\": label, \\"n\\": n,\\n",\n    "               \\"format_rate\\": sum(r[\\"format_ok\\"] for r in rows)/n,\\n",\n    "               \\"content_rate\\": sum(r[\\"content_ok\\"] for r in rows)/n,\\n",\n    "               \\"both_rate\\": sum(r[\\"both_ok\\"] for r in rows)/n,\\n",\n    "               \\"seconds\\": round(time.time()-t0, 1)}\\n",\n    "    (WORK / f\\"eval_{label}.json\\").write_text(json.dumps({\\"summary\\": summary, \\"rows\\": rows}, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n",\n    "    print(json.dumps(summary, ensure_ascii=False))\\n",\n    "    return summary, rows\\n",\n    "\\n",\n    "before_summary, before_rows = evaluate(model, \\"before\\")\\n",\n    "# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む\\n",\n    "display(pd.DataFrame([{\\"質問\\": r[\\"question\\"], \\"正解の needs_review\\": r[\\"gold_needs_review\\"], \\"学習前の出力\\": r[\\"raw\\"],\\n",\n    "                       \\"形式\\": mark(r[\\"format_ok\\"]), \\"形式の失敗理由\\": r[\\"format_reason\\"], \\"内容\\": mark(r[\\"content_ok\\"])} for r in before_rows]))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "d203e72c",\n   "metadata": {},\n   "source": [\n    "## 4. LoRA 学習（T4 で 4〜5 分）\\n",\n    "\\n",\n    "### LoRA と QLoRA の違い\\n",\n    "\\n",\n    "- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。\\n",\n    "- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。\\n",\n    "\\n",\n    "QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。\\n",\n    "\\n",\n    "| 観点 | LoRA（量子化なし） | QLoRA（4bit） |\\n",\n    "| --- | --- | --- |\\n",\n    "| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |\\n",\n    "| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |\\n",\n    "| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |\\n",\n    "| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |\\n",\n    "\\n",\n    "QLoRA を実装するときの注意:\\n",\n    "\\n",\n    "- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。\\n",\n    "- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。\\n",\n    "- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。\\n",\n    "- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。\\n",\n    "- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。\\n",\n    "\\n",\n    "この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。\\n",\n    "\\n",\n    "### 学習の実行\\n",\n    "\\n",\n    "**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。\\n",\n    "\\n",\n    "- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。\\n",\n    "- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。\\n",\n    "- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。\\n",\n    "\\n",\n    "**見るところ**\\n",\n    "\\n",\n    "- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。\\n",\n    "- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。\\n",\n    "- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。\\n",\n    "\\n",\n    "**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。\\n",\n    "\\n",\n    "**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "0a99a43f",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "from peft import LoraConfig\\n",\n    "from trl import SFTTrainer, SFTConfig\\n",\n    "from datasets import Dataset\\n",\n    "\\n",\n    "def to_chat(example):\\n",\n    "    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う\\n",\n    "    return {\\"messages\\": build_messages(example) + [{\\"role\\": \\"assistant\\", \\"content\\": example[\\"target\\"]}]}\\n",\n    "\\n",\n    "train_ds = Dataset.from_list([to_chat(e) for e in train])\\n",\n    "valid_ds = Dataset.from_list([to_chat(e) for e in valid])\\n",\n    "\\n",\n    "model.config.use_cache = False\\n",\n    "# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある\\n",\n    "lora = LoraConfig(r=CONFIG[\\"lora_r\\"], lora_alpha=CONFIG[\\"lora_alpha\\"], lora_dropout=0.05, bias=\\"none\\", task_type=\\"CAUSAL_LM\\",\\n",\n    "                  target_modules=[\\"q_proj\\", \\"k_proj\\", \\"v_proj\\", \\"o_proj\\", \\"gate_proj\\", \\"up_proj\\", \\"down_proj\\"])\\n",\n    "USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある\\n",\n    "sft = SFTConfig(output_dir=str(WORK / \\"ckpt\\"), num_train_epochs=CONFIG[\\"num_train_epochs\\"], per_device_train_batch_size=4, gradient_accumulation_steps=2,\\n",\n    "                learning_rate=CONFIG[\\"lr\\"], lr_scheduler_type=\\"cosine\\", warmup_steps=CONFIG[\\"warmup_steps\\"], logging_steps=10, save_steps=10, save_total_limit=2,\\n",\n    "                fp16=USE_AMP, max_length=CONFIG[\\"max_seq_len\\"], packing=False, report_to=\\"none\\", seed=CONFIG[\\"seed\\"])\\n",\n    "trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)\\n",\n    "trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合\\n",\n    "t0 = time.time()\\n",\n    "trainer.train()  # 途中で止まった場合の再開は 4b を参照\\n",\n    "print(f\\"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}\\")\\n",\n    "ADAPTER_DIR = WORK / \\"adapter\\"\\n",\n    "trainer.model.save_pretrained(ADAPTER_DIR)\\n",\n    "if USE_DRIVE:\\n",\n    "    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG[\\"drive_dir\\"]) / \\"adapter\\", dirs_exist_ok=True)\\n",\n    "tuned = trainer.model\\n",\n    "tuned.config.use_cache = True\\n",\n    "_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "89fc48b5",\n   "metadata": {},\n   "source": [\n    "### 4b. 学習が止まった・できなかった場合\\n",\n    "\\n",\n    "**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。\\n",\n    "\\n",\n    "**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "4970b14e",\n   "metadata": {},\n   "source": [\n    "## 5. 学習後評価（学習前と同じ test・同じ生成設定）\\n",\n    "\\n",\n    "**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。\\n",\n    "\\n",\n    "**見るところ**\\n",\n    "\\n",\n    "- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。\\n",\n    "- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。\\n",\n    "- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。\\n",\n    "  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。\\n",\n    "  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。\\n",\n    "\\n",\n    "学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。\\n",\n    "\\n",\n    "seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "90ecb294",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "after_summary, after_rows = evaluate(tuned, \\"after\\")\\n",\n    "compare = pd.DataFrame([before_summary, after_summary]).set_index(\\"label\\")[[\\"n\\", \\"format_rate\\", \\"content_rate\\", \\"both_rate\\", \\"seconds\\"]]\\n",\n    "display(compare)\\n",\n    "# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）\\n",\n    "regressed = [b[\\"question\\"] for b, a in zip(before_rows, after_rows) if b[\\"both_ok\\"] and not a[\\"both_ok\\"]]\\n",\n    "print(\\"退行:\\", regressed)\\n",\n    "\\n",\n    "# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる\\n",\n    "detail = pd.DataFrame([{\\"質問\\": b[\\"question\\"], \\"正解の needs_review\\": b[\\"gold_needs_review\\"],\\n",\n    "                        \\"学習前の出力\\": b[\\"raw\\"], \\"学習前 形式/内容\\": f\\"{mark(b[\'format_ok\'])}/{mark(b[\'content_ok\'])}\\",\\n",\n    "                        \\"学習後の出力\\": a[\\"raw\\"], \\"学習後 形式/内容\\": f\\"{mark(a[\'format_ok\'])}/{mark(a[\'content_ok\'])}\\"}\\n",\n    "                       for b, a in zip(before_rows, after_rows)])\\n",\n    "display(detail)"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "6e27ea5e",\n   "metadata": {},\n   "source": [\n    "## 6. 知識を問う質問（形式を学んでも知識は保証されない）\\n",\n    "\\n",\n    "**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。\\n",\n    "\\n",\n    "応答を 3 つに分類します。\\n",\n    "\\n",\n    "| 分類 | 意味 |\\n",\n    "| --- | --- |\\n",\n    "| `format_error` | JSON の形式を守れていない（学習前に多い） |\\n",\n    "| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |\\n",\n    "| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |\\n",\n    "\\n",\n    "**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。\\n",\n    "\\n",\n    "学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "51e56a48",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "KNOWLEDGE_QUESTIONS = [\\n",\n    "    \\"この講座の最終課題の締切はいつですか\\",       # FAQ に無い（架空）\\n",\n    "    \\"担当教員の名前は何ですか\\",                     # FAQ に無い\\n",\n    "    \\"次回の講義のテーマは何ですか\\",                 # FAQ に無い\\n",\n    "    \\"課題の提出期限はいつですか\\",                   # FAQ にある（対照）\\n",\n    "]\\n",\n    "def classify(raw):\\n",\n    "    ok, _, obj = check_format(raw)\\n",\n    "    if not ok or obj is None: return \\"format_error\\"\\n",\n    "    return \\"needs_review\\" if obj[\\"needs_review\\"] else \\"asserted\\"\\n",\n    "\\n",\n    "knowledge_rows = []\\n",\n    "for q in KNOWLEDGE_QUESTIONS:\\n",\n    "    ex = make_example(q, \\"\\", True)\\n",\n    "    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る\\n",\n    "    with tuned.disable_adapter():\\n",\n    "        before_raw = generate(tuned, ex)\\n",\n    "    after_raw = generate(tuned, ex)\\n",\n    "    knowledge_rows.append({\\"質問\\": q, \\"FAQ にある\\": q in gold, \\"学習前の分類\\": classify(before_raw), \\"学習前の出力\\": before_raw,\\n",\n    "                 \\"学習後の分類\\": classify(after_raw), \\"学習後の出力\\": after_raw})\\n",\n    "display(pd.DataFrame(knowledge_rows))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "abc0c0b6",\n   "metadata": {},\n   "source": [\n    "## 7. 比較表と提出\\n",\n    "\\n",\n    "**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。\\n",\n    "\\n",\n    "**提出するもの**\\n",\n    "\\n",\n    "- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）\\n",\n    "- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）\\n",\n    "- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）\\n",\n    "- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "513753a1",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "result = {\\n",\n    "    \\"config\\": {k: v for k, v in CONFIG.items() if k not in (\\"work_dir\\", \\"drive_dir\\")},\\n",\n    "    \\"versions\\": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},\\n",\n    "    \\"method\\": \\"LoRA（量子化なし）\\",\\n",\n    "    \\"before\\": before_summary, \\"after\\": after_summary,\\n",\n    "    \\"regressed\\": regressed,\\n",\n    "    \\"knowledge\\": knowledge_rows,\\n",\n    "    \\"executed_training_myself\\": True,   # 講師の結果を使った場合は False にする\\n",\n    "}\\n",\n    "(WORK / \\"result.json\\").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n",\n    "print(json.dumps({k: v for k, v in result.items() if k != \\"knowledge\\"}, ensure_ascii=False, indent=2))\\n",\n    "if USE_DRIVE:\\n",\n    "    import shutil\\n",\n    "    for name in [\\"result.json\\", \\"eval_before.json\\", \\"eval_after.json\\", \\"test.jsonl\\", \\"gold.json\\"]:\\n",\n    "        shutil.copy(WORK / name, Path(CONFIG[\\"drive_dir\\"]) / name)\\n",\n    "    print(\\"Drive に保存しました:\\", CONFIG[\\"drive_dir\\"])"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "66e678f1",\n   "metadata": {},\n   "source": [\n    "## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）\\n",\n    "\\n",\n    "**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。\\n",\n    "\\n",\n    "- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。\\n",\n    "- 4bit の読み込みは `device_map=\\"auto\\"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。\\n",\n    "- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。\\n",\n    "\\n",\n    "**見るところ**\\n",\n    "\\n",\n    "- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。\\n",\n    "- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。良くなる方向に変わる質問もあれば、`needs_review` が `\\"false\\"`（文字列）になって型検査で落ちる、根拠なしの質問に「はい」「なし」と断定する、といった出力も見られます。方向が一定でないことを確かめてください。"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "a4bd62ce",\n   "metadata": {},\n   "outputs": [],\n   "source": [\n    "%pip install -q \\"bitsandbytes==0.50.*\\"\\n",\n    "from transformers import BitsAndBytesConfig\\n",\n    "\\n",\n    "bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\\"nf4\\",   # 保持精度: 4bit（NF4）\\n",\n    "                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）\\n",\n    "                         bnb_4bit_use_double_quant=True)\\n",\n    "model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG[\\"base_model\\"], revision=CONFIG[\\"revision\\"], quantization_config=bnb, device_map=\\"auto\\")\\n",\n    "print(\\"重みのメモリ(GB) | 量子化なし（32bit）:\\", BASE_MEMORY_GB, \\"| 4bit:\\", round(model_4bit.get_memory_footprint() / 1e9, 2))\\n",\n    "\\n",\n    "# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響\\n",\n    "q_summary, q_rows = evaluate(model_4bit, \\"before_4bit\\")\\n",\n    "display(pd.DataFrame([before_summary, q_summary]).set_index(\\"label\\")[[\\"n\\", \\"format_rate\\", \\"content_rate\\", \\"both_rate\\", \\"seconds\\"]])\\n",\n    "display(pd.DataFrame([{\\"質問\\": r[\\"question\\"], \\"正解の needs_review\\": r[\\"gold_needs_review\\"], \\"4bit・学習前の出力\\": r[\\"raw\\"],\\n",\n    "                       \\"形式/内容\\": f\\"{mark(r[\'format_ok\'])}/{mark(r[\'content_ok\'])}\\", \\"形式の失敗理由\\": r[\\"format_reason\\"]} for r in q_rows]))"\n   ]\n  }\n ],\n "metadata": {\n  "accelerator": "GPU",\n  "colab": {\n   "gpuType": "T4",\n   "provenance": []\n  },\n  "kernelspec": {\n   "display_name": "Python 3",\n   "language": "python",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python"\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}\n',encoding="utf-8")
(ROOT/"kaggle_runner.py").write_text('"""Run one official Day2 configuration in a fresh Kaggle GPU process."""\nimport argparse\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--epochs\', type=int, choices=[20,10], required=True)\n    parser.add_argument(\'--root\', required=True)\n    args = parser.parse_args()\n    root = Path(args.root)\n    work = root / f\'epochs-{args.epochs}\'\n    work.mkdir(parents=True, exist_ok=True)\n    if (work/\'result.json\').exists():\n        raise RuntimeError(\'Completed result already exists. Preserve the result and use a fresh run folder.\')\n    nb = json.loads((root/\'official.ipynb\').read_text(encoding=\'utf-8\'))\n    source = lambda i: \'\'.join(nb[\'cells\'][i][\'source\'])\n    env = {\'display\':print}\n    print(\'START\', args.epochs, time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),flush=True)\n    try:\n        import torch\n        import transformers, peft, trl, accelerate, datasets\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'No CUDA GPU allocated; no model loading or training started.\')\n        torch.set_num_threads(2)\n        torch.set_num_interop_threads(1)\n        runtime = {\'python\':sys.version,\'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n                   \'gpu_memory_bytes\':torch.cuda.get_device_properties(0).total_memory,\n                   \'versions\':{m.__name__:m.__version__ for m in [transformers,peft,trl,accelerate,datasets]}}\n        (work/\'runtime.json\').write_text(json.dumps(runtime,indent=2),encoding=\'utf-8\')\n        print(\'RUNTIME\',json.dumps(runtime),flush=True)\n        expected = {\'transformers\':\'5.18.0\',\'peft\':\'0.21.1\',\'trl\':\'1.14.2\',\'accelerate\':\'1.15.0\',\'datasets\':\'5.1.0\'}\n        for name, version in expected.items():\n            if importlib.metadata.version(name) != version:\n                raise RuntimeError(f\'{name} must be {version}\')\n        env.update(torch=torch,transformers=transformers,peft=peft,trl=trl)\n        # The reviewed configuration cell creates its work directory before any data/model is loaded.\n        config_cell = source(5).replace(\'"work_dir": "/content/qlora_lab"\', \'"work_dir": \'+repr(str(work)))\n        config_cell = config_cell.replace(\'"num_train_epochs": 20,\',\'"num_train_epochs": \'+str(args.epochs)+\',\',1)\n        exec(compile(config_cell,\'official_config\',\'exec\'),env)\n        if env[\'CONFIG\'][\'work_dir\'] != str(work) or env[\'CONFIG\'][\'num_train_epochs\'] != args.epochs:\n            raise RuntimeError(\'Configuration substitution failed\')\n        for i in [7,9,10]:\n            print(\'CELL\',i,flush=True)\n            exec(compile(source(i),f\'official_cell_{i}\',\'exec\'),env)\n        training = source(12).replace(\'trainer.train()  #\',\'trainer.train()\\nTRAINING_SECONDS = time.time() - t0\\n#\',1)\n        print(\'TRAINING\',args.epochs,flush=True)\n        exec(compile(training,\'official_training\',\'exec\'),env)\n        for i in [15,17,19]:\n            print(\'CELL\',i,flush=True)\n            exec(compile(source(i),f\'official_cell_{i}\',\'exec\'),env)\n        result = env[\'result\']\n        losses = [r for r in env[\'trainer\'].state.log_history if \'loss\' in r]\n        result[\'training_metrics\'] = {\'training_seconds\':env[\'TRAINING_SECONDS\'],\n             \'final_logged_loss\':losses[-1][\'loss\'] if losses else None,\n             \'final_logged_loss_step\':losses[-1][\'step\'] if losses else None,\n             \'global_step\':env[\'trainer\'].state.global_step,\n             \'loss_definition\':\'Last logged training loss, averaged over the final logging interval.\'}\n        result[\'runtime\'] = runtime\n        result[\'provenance\'] = {\'kind\':\'actual_kaggle_training\',\'official_commit\':\'fae899a409e2c99feee9d44a3789f936f5fc64ee\',\n             \'requested_changed_setting\':\'num_train_epochs\',\'run_epochs\':args.epochs,\n             \'common_adaptations\':[\'Kaggle output directory\',\'Only first allocated GPU visible\',\'CPU threads 2\'],\n             \'notes\':\'No course model, data, batch size, learning rate, rank, generation, or evaluation criteria were replaced.\'}\n        (work/\'result.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n        (work/\'training_log_history.json\').write_text(json.dumps(env[\'trainer\'].state.log_history,indent=2),encoding=\'utf-8\')\n        print(\'COMPLETE\',args.epochs,json.dumps(result[\'training_metrics\']),flush=True)\n    except Exception as exc:\n        (work/\'failure.json\').write_text(json.dumps({\'type\':type(exc).__name__,\'message\':str(exc),\'traceback\':traceback.format_exc(),\'training_completed\':False},ensure_ascii=False,indent=2),encoding=\'utf-8\')\n        raise\n\nif __name__ == \'__main__\':\n    main()\n',encoding="utf-8")


In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
for epochs in [20, 10]:
    log_path = ROOT / f'epochs-{epochs}.log'
    with log_path.open('w',encoding='utf-8') as log:
        process = subprocess.Popen([sys.executable, '-u', str(ROOT/'kaggle_runner.py'), '--epochs', str(epochs), '--root', str(ROOT)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line); log.flush()
        if process.wait() != 0:
            raise RuntimeError(f'{epochs} epochs failed. Inspect {log_path} and failure.json. No completed comparison is claimed.')


In [ ]:
results = [json.loads((ROOT/f'epochs-{e}/result.json').read_text(encoding='utf-8')) for e in [20,10]]
baseline, changed = results
diff = {k:(baseline['config'].get(k),changed['config'].get(k)) for k in set(baseline['config'])|set(changed['config']) if baseline['config'].get(k)!=changed['config'].get(k)}
assert diff == {'num_train_epochs':(20,10)}, diff
assert all(r['executed_training_myself'] is True for r in results)
assert (ROOT/'epochs-20/test.jsonl').read_bytes() == (ROOT/'epochs-10/test.jsonl').read_bytes()
rows = [{'num_train_epochs':r['config']['num_train_epochs'], 'format_rate':r['after']['format_rate'], 'content_rate':r['after']['content_rate'], **r['training_metrics']} for r in results]
(ROOT/'comparison.json').write_text(json.dumps({'only_config_difference':diff,'evaluation_data_identical':True,'rows':rows},ensure_ascii=False,indent=2),encoding='utf-8')
with (ROOT/'comparison.csv').open('w',encoding='utf-8',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
print(json.dumps(rows,ensure_ascii=False,indent=2))
archive = shutil.make_archive('/kaggle/working/aie2026-day2-results','zip',ROOT)
print('Saved actual training outputs:',archive)
